In [ ]:
!pip install -q openai pydantic python-dotenv sentence-transformers sqlalchemy "psycopg[binary]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Env vars BEFORE importing the package (llm.client and db.session read them at import time).
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
os.environ['LLM_MODEL'] = userdata.get('LLM_MODEL')            # e.g. z-ai/glm-4.6
os.environ['LLM_MAX_TOKENS'] = '8192'
os.environ['DATABASE_URL'] = userdata.get('SUPABASE_DB_URL')   # only needed at import; not queried

DRIVE_DIR = "/content/drive/MyDrive/Bachelor's Project/lms_checkpoints"  # atoms.json + llm_cache live here
os.makedirs(DRIVE_DIR, exist_ok=True)

In [ ]:
# Project source. Option A: copy of src/ in Drive. Option B: fresh git clone.
# If neither matches your setup, set SRC_ROOT to any folder that contains
# bachelor_s_project_i_lms/ (the package directory with graph/, llm/, ...).
import sys

SRC_DRIVE = "/content/drive/MyDrive/Bachelor's Project/src"
if os.path.isdir(SRC_DRIVE):
    SRC_ROOT = SRC_DRIVE
else:
    !git clone https://github.com/itsmohsentaheri/Bachelor-s-Project-I-LMS.git /content/repo
    SRC_ROOT = "/content/repo/src"

sys.path.insert(0, SRC_ROOT)
print("source:", SRC_ROOT)

In [ ]:
# Stage inputs: atoms.json from Drive, and seed the LLM cache with any
# partial Phase-5 verdicts from earlier interrupted runs (free resume).
import shutil
from pathlib import Path

import bachelor_s_project_i_lms.graph.cross_branch as cb

ATOMS = Path(DRIVE_DIR) / 'atoms.json'
assert ATOMS.exists(), f'{ATOMS} not found — copy atoms.json into lms_checkpoints first'

drive_cache = Path(DRIVE_DIR) / 'llm_cache'
if drive_cache.exists():
    cb.CACHE_DIR.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in drive_cache.glob('crossbranch_*.json'):
        shutil.copy(f, cb.CACHE_DIR / f.name)
        n += 1
print(f'seeded cache: {n} crossbranch verdicts')

In [ ]:
# The run. Every verdict is cached, so a 402 / timeout / runtime disconnect
# just means: re-run this cell (plus cell 4) and only unjudged pairs are paid for.
edges, hist = cb.run(atoms_path=ATOMS)

In [ ]:
# Persist everything back to Drive: the edges, and the full verdict cache.
shutil.copy(cb.CROSS_EDGES_PATH, Path(DRIVE_DIR) / 'cross_edges.json')

drive_cache = Path(DRIVE_DIR) / 'llm_cache'
drive_cache.mkdir(parents=True, exist_ok=True)
n = 0
for f in cb.CACHE_DIR.glob('crossbranch_*.json'):
    shutil.copy(f, drive_cache / f.name)
    n += 1
print(f'copied back: {len(edges)} cross-branch edges, {n} cached verdicts')
print('histogram:', hist)